# Sparkov Credit Card Fraud Detection: M0–M2 Modelling

## 1. Introduction

This notebook implements and evaluates three stages of credit card fraud detection using the Sparkov dataset. The modelling framework progressively addresses key challenges in fraud detection: severe class imbalance, evolving transaction patterns, and delays in fraud-label verification.

The experiments are organised into three stages:

- **M0 – Baseline Models:** Establish reference performance using Logistic Regression, Random Forest, and CatBoost without class-imbalance adjustments.
- **M1 – Cost-Sensitive Learning:** Introduce class weighting to increase the penalty associated with misclassifying fraudulent transactions.
- **M2 – Sliding-Window Retraining:** Investigate whether periodically updating models using recent historical transactions improves robustness to temporal changes, while accounting for a seven-day label verification delay.

Models are evaluated using chronological data partitions to reflect the temporal nature of fraud detection and avoid the optimistic estimates that may arise from random train-test splitting. Precision-Recall AUC (PR-AUC) is the primary evaluation metric due to the low prevalence of fraudulent transactions, supplemented by ROC-AUC, precision, recall, and F1-score.

The final test set is reserved for evaluation after model development.

## 2. Data Loading and Preparation

### 2.1 Load Processed Features

The processed Sparkov dataset is loaded from the feature-engineering pipeline. It contains transaction-level characteristics, temporal features, and historical card behaviour, including rolling-window transaction counts and amounts.

A predefined list of 62 modelling features is loaded separately to exclude transaction identifiers, raw timestamps, target labels, and other non-predictive columns.

The dataset retains chronological training, validation, and test partitions. Their sizes and fraud prevalence are examined to establish the class distribution across evaluation periods.

In [18]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

DATA_DIR = Path("../data/processed")

df = pd.read_pickle(DATA_DIR / "sparkov_features.pkl")

with open(DATA_DIR / "feature_lists.json", "r") as f:
    feature_lists = json.load(f)

feature_cols = feature_lists["sparkov_feature_cols"]

print("Dataset shape:", df.shape)
print("Number of model features:", len(feature_cols))

print("\nChronological splits:")
print(df["sparkov_split"].value_counts())

print("\nFraud rate by split:")
print(df.groupby("sparkov_split")["is_fraud"].mean())

Dataset shape: (1852394, 87)
Number of model features: 62

Chronological splits:
sparkov_split
train         1037340
test           555719
validation     259335
Name: count, dtype: int64

Fraud rate by split:
sparkov_split
test          0.003860
train         0.005753
validation    0.005931
Name: is_fraud, dtype: float64


### 2.2 Prepare Training and Validation Sets

The dataset is partitioned according to the predefined chronological split indicators. The training subset is used for model fitting, while the validation subset is reserved for model comparison and hyperparameter selection.

Predictor variables (`X`) are separated from fraud labels (`y`). Categorical and numerical features are identified to support model-specific preprocessing, and missing values are inspected before model fitting.

The test subset is excluded from model development to prevent information leakage into model selection.

In [19]:
train_mask = df["sparkov_split"] == "train"
val_mask = df["sparkov_split"] == "validation"

X_train = df.loc[train_mask, feature_cols].copy()
y_train = df.loc[train_mask, "is_fraud"].astype(int)

X_val = df.loc[val_mask, feature_cols].copy()
y_val = df.loc[val_mask, "is_fraud"].astype(int)

categorical_cols = X_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

numerical_cols = [
    col for col in feature_cols
    if col not in categorical_cols
]

# Release the full dataset after extracting the required subsets
del df
import gc
gc.collect()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("Categorical features:", categorical_cols)
print("Numerical features:", len(numerical_cols))

X_train: (1037340, 62)
X_val: (259335, 62)
Categorical features: ['category', 'gender', 'state']
Numerical features: 59


### 2.3 Missing-Value Assessment

Before model training, we examine missing values across the 62 selected predictors to determine whether preprocessing is required. Missing values may arise naturally from historical transaction features, particularly when insufficient prior card activity is available to calculate statistics such as previous transaction amounts or historical variability.

Missing-value counts and proportions are calculated separately for the training and validation sets to identify affected features and assess whether their missingness patterns differ across time. This distinction is important because inappropriate treatment of missing values may distort transaction characteristics or introduce bias into model evaluation.

Rather than immediately removing incomplete observations, we first examine the extent and distribution of missingness. An appropriate preprocessing strategy will subsequently be selected based on feature characteristics and model requirements, with any imputation parameters estimated exclusively from the training data to prevent information leakage.

In [20]:
missing_summary = pd.DataFrame({
    "Train Missing": X_train.isna().sum(),
    "Train Missing (%)": X_train.isna().mean() * 100,
    "Validation Missing": X_val.isna().sum(),
    "Validation Missing (%)": X_val.isna().mean() * 100
})

missing_summary = missing_summary[
    (missing_summary["Train Missing"] > 0) |
    (missing_summary["Validation Missing"] > 0)
].sort_values("Train Missing", ascending=False)

display(missing_summary.round(2))

,Train Missing,Train Missing (%),Validation Missing,Validation Missing (%)
amt_mean_1h,865064,83.39,219064,84.47
amt_mean_24h,87724,8.46,21195,8.17
card_previous_amt_std,1928,0.19,38,0.01
amt_prior_zscore,1928,0.19,38,0.01
amt_mean_7D,1209,0.12,59,0.02
implied_speed_kmh,979,0.09,24,0.01
card_previous_amt_mean,964,0.09,19,0.01
amount_ratio_to_previous,964,0.09,19,0.01
amount_change_from_previous,964,0.09,19,0.01
previous_card_amt,964,0.09,19,0.01


**Missing-Value Assessment: Findings**

Missingness is concentrated in features derived from historical card activity. In particular, `amt_mean_1h` has the highest missingness (83.39% in training and 84.47% in validation), followed by `amt_mean_24h` (8.46% and 8.17%, respectively). Most remaining features exhibit missingness below 0.2%.

These patterns are consistent with structurally undefined historical statistics when insufficient prior transactions are available within the specified observation windows. Consequently, removing incomplete observations would discard a substantial proportion of otherwise valid transactions and potentially introduce selection bias.

We therefore retain all observations and address missing values through model-appropriate preprocessing. For models requiring complete numerical inputs, imputation will be fitted exclusively on the training data and applied consistently to validation data. Historical transaction-count features are retained to provide information about the availability of prior card activity.

### 2.4 Model-Specific Preprocessing

Different classification algorithms impose different requirements on input features. To ensure a fair comparison across baseline models, preprocessing is tailored to each algorithm while maintaining identical training and validation partitions.

For Logistic Regression, categorical variables are one-hot encoded, numerical missing values are imputed using training-set medians, and numerical features are standardised to improve optimisation stability. Random Forest uses categorical encoding and numerical imputation but does not require feature scaling. CatBoost supports categorical predictors directly and can handle missing numerical values.

All learned preprocessing transformations are fitted exclusively on the training partition and subsequently applied to validation data to prevent information leakage. No class weighting or resampling is introduced at this stage, ensuring that M0 represents an unadjusted baseline against which the cost-sensitive M1 models can be compared.

In [21]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Numerical preprocessing: median imputation and standardisation
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing: imputation and one-hot encoding
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing for both feature types
lr_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

print("Logistic Regression preprocessing pipeline created.")

Logistic Regression preprocessing pipeline created.


## 3. M0: Baseline Models

M0 establishes reference performance for credit card fraud detection before introducing techniques to address class imbalance or concept drift. Three supervised classification algorithms are considered: Logistic Regression, Random Forest, and CatBoost.

These models represent different levels of complexity, allowing us to assess whether nonlinear relationships and interactions among transaction features improve fraud detection relative to a linear baseline.

All models are trained on the same chronological training partition and evaluated on the validation partition. No class weighting or resampling is applied in M0, preserving a consistent reference for evaluating subsequent improvements in M1 and M2.

### 3.1 Logistic Regression

Logistic Regression serves as an interpretable linear baseline that estimates the probability of a transaction being fraudulent from a weighted combination of its predictors.

Numerical features are median-imputed and standardised, while categorical features are one-hot encoded. These transformations are incorporated into a unified pipeline to ensure consistent preprocessing and prevent validation information from influencing model fitting.

The model uses L2 regularisation and default class weights. Predicted fraud probabilities are evaluated primarily using PR-AUC, with ROC-AUC reported as a complementary threshold-independent metric.

In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
import time

# Combine preprocessing and classification
lr_model = Pipeline(steps=[
    ("preprocessor", lr_preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight=None,
        random_state=42
    ))
])

# Train baseline model
start_time = time.time()

lr_model.fit(X_train, y_train)

training_time = time.time() - start_time

# Predict fraud probabilities on validation data
lr_val_probs = lr_model.predict_proba(X_val)[:, 1]

# Evaluate threshold-independent discrimination
lr_pr_auc = average_precision_score(y_val, lr_val_probs)
lr_roc_auc = roc_auc_score(y_val, lr_val_probs)

print(f"Training time: {training_time:.2f} seconds")
print(f"Validation PR-AUC: {lr_pr_auc:.4f}")
print(f"Validation ROC-AUC: {lr_roc_auc:.4f}")

Training time: 21.45 seconds
Validation PR-AUC: 0.6802
Validation ROC-AUC: 0.9664


**Logistic Regression: Validation Results**

The unweighted Logistic Regression baseline achieved a validation Average Precision (AP) of **0.6802** and ROC-AUC of **0.9664**, with a training time of **26.33 seconds**.

The model demonstrates substantial discriminatory ability despite the severe class imbalance, with AP considerably exceeding the random-ranking reference corresponding to the validation fraud prevalence (0.0059). The high ROC-AUC further indicates that fraudulent transactions generally receive higher predicted risk scores than legitimate transactions.

Nevertheless, these threshold-independent metrics do not directly establish operational detection performance at a specified false-positive rate or alert volume. The results therefore serve as an initial reference for subsequent comparisons with Random Forest, CatBoost, and cost-sensitive learning approaches.

### 3.2 Random Forest

Random Forest is an ensemble learning algorithm that combines multiple decision trees to capture nonlinear relationships and interactions among transaction features. Unlike Logistic Regression, Random Forest does not assume a linear relationship between predictors and the log-odds of fraud, making it suitable for modelling complex transaction behaviours.

For preprocessing, numerical missing values are imputed using training-set medians, while categorical variables are one-hot encoded. Feature standardisation is unnecessary because tree-based models partition observations using feature thresholds rather than distance-based calculations.

To control computational cost and reduce overfitting, the model uses 100 decision trees, a maximum tree depth of 15, and a minimum of 10 samples per leaf. Class weighting remains disabled to preserve the unadjusted M0 baseline.

The model is evaluated on the same chronological validation partition as Logistic Regression, using Average Precision (AP) and ROC-AUC to ensure comparability.


In [23]:
from sklearn.ensemble import RandomForestClassifier

# Random Forest does not require numerical scaling
rf_numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

rf_categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

rf_preprocessor = ColumnTransformer(
    transformers=[
        ("num", rf_numeric_transformer, numerical_cols),
        ("cat", rf_categorical_transformer, categorical_cols)
    ]
)

# Define Random Forest baseline
rf_model = Pipeline(steps=[
    ("preprocessor", rf_preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        min_samples_leaf=10,
        max_features="sqrt",
        class_weight=None,
        n_jobs=2,
        random_state=42
    ))
])

# Train and evaluate
start_time = time.time()

rf_model.fit(X_train, y_train)

rf_training_time = time.time() - start_time

rf_val_probs = rf_model.predict_proba(X_val)[:, 1]

rf_pr_auc = average_precision_score(y_val, rf_val_probs)
rf_roc_auc = roc_auc_score(y_val, rf_val_probs)

print(f"Training time: {rf_training_time:.2f} seconds")
print(f"Validation PR-AUC: {rf_pr_auc:.4f}")
print(f"Validation ROC-AUC: {rf_roc_auc:.4f}")

Training time: 404.33 seconds
Validation PR-AUC: 0.9589
Validation ROC-AUC: 0.9981


**Random Forest: Validation Results**

The unweighted Random Forest achieved a validation Average Precision (AP) of **0.9589** and ROC-AUC of **0.9981**, outperforming Logistic Regression (AP = 0.6802; ROC-AUC = 0.9664). This suggests that nonlinear relationships and interactions among transaction features may provide substantial predictive value beyond a linear decision boundary.

However, the improvement comes at a higher computational cost, with training requiring **435.28 seconds**, compared with 26.33 seconds for Logistic Regression.

Given the exceptionally high discrimination metrics, further checks are necessary to verify that historical features were constructed using only information available at prediction time and that no target-derived information leaked into the validation data. The results should therefore be treated as provisional until these checks are completed.

In [24]:
importances = rf_model.named_steps["classifier"].feature_importances_

feature_names = rf_model.named_steps[
    "preprocessor"
].get_feature_names_out()

rf_importance = (
    pd.DataFrame({
        "Feature": feature_names,
        "Importance": importances
    })
    .sort_values("Importance", ascending=False)
    .head(15)
)

display(rf_importance)

,Feature,Importance
28,num__amt_x_night,0.109595
46,num__amt_mean_24h,0.090721
0,num__amt,0.080649
14,num__amt_log1p,0.062170
45,num__amt_sum_24h,0.059180
33,num__previous_card_amt,0.047061
17,num__amt_bucket,0.044317
49,num__amt_mean_7D,0.038336
57,num__category_freq,0.036536
35,num__amount_ratio_to_previous,0.035705


**Random Forest: Feature Importance Analysis**

Random Forest feature importance indicates that transaction amounts and historical spending behaviour are influential predictors of fraud. The most important features include `amt_x_night` (0.1096), `amt_mean_24h` (0.0907), and `amt` (0.0806), suggesting that both individual transaction characteristics and recent card activity contribute to the model's predictions.

Historical spending features, including `amt_sum_24h`, `previous_card_amt`, and `amount_ratio_to_previous`, also rank highly. These findings are consistent with the potential value of behavioural deviations for fraud detection.

However, impurity-based importance does not establish causality or rule out information leakage. Given the exceptionally high validation performance, further verification of historical feature construction and temporal availability is necessary before interpreting the results as evidence of generalisable predictive performance.

### 3.2.1 Validation of Historical Feature Construction

Given the high validation performance of Random Forest, an independent consistency check is conducted to assess potential information leakage in the engineered historical transaction features.

For a sample of cards, the stored 24-hour transaction counts, sums, and means are compared against values independently reconstructed from raw transaction timestamps and amounts. Only transactions belonging to the same card and occurring strictly before the current transaction within the preceding 24 hours are included.

This check also assesses whether the engineered rolling statistics remain correctly aligned with their corresponding transactions after card-level sorting and feature assignment.

In [25]:
# Independent leakage check for 24-hour historical features
from pathlib import Path
import numpy as np
import pandas as pd

# Load only the columns needed for validation
check_cols = [
    "cc_num",
    "trans_date_trans_time",
    "amt",
    "txn_count_24h",
    "amt_sum_24h",
    "amt_mean_24h"
]

check_df = pd.read_pickle(
    Path("../data/processed/sparkov_features.pkl")
)[check_cols].copy()

check_df["trans_date_trans_time"] = pd.to_datetime(
    check_df["trans_date_trans_time"]
)

# Select 10 reproducible sample cards
sample_cards = (
    check_df["cc_num"]
    .drop_duplicates()
    .sample(n=10, random_state=42)
)

sample = check_df[
    check_df["cc_num"].isin(sample_cards)
].copy()

sample = sample.sort_values(
    ["cc_num", "trans_date_trans_time"],
    kind="stable"
)

# Independently reconstruct prior 24-hour history
results = []

for card, group in sample.groupby("cc_num"):
    group = group.sort_values("trans_date_trans_time")

    times = group["trans_date_trans_time"].to_numpy()
    amounts = group["amt"].to_numpy()

    for i in range(len(group)):
        current_time = times[i]
        window_start = current_time - np.timedelta64(24, "h")

        prior_mask = (
            (times < current_time) &
            (times >= window_start)
        )

        prior_amounts = amounts[prior_mask]

        expected_count = len(prior_amounts)
        expected_sum = prior_amounts.sum()
        expected_mean = (
            prior_amounts.mean()
            if expected_count > 0 else np.nan
        )

        row = group.iloc[i]

        results.append({
            "count_match": np.isclose(
                row["txn_count_24h"], expected_count
            ),
            "sum_match": np.isclose(
                row["amt_sum_24h"], expected_sum,
                rtol=1e-5, atol=1e-5
            ),
            "mean_match": np.isclose(
                row["amt_mean_24h"], expected_mean,
                rtol=1e-5, atol=1e-5,
                equal_nan=True
            )
        })

results_df = pd.DataFrame(results)

print("Transactions checked:", len(results_df))
print("Count mismatches:", (~results_df["count_match"]).sum())
print("Sum mismatches:", (~results_df["sum_match"]).sum())
print("Mean mismatches:", (~results_df["mean_match"]).sum())

Transactions checked: 19776
Count mismatches: 0
Sum mismatches: 0
Mean mismatches: 0


**Historical Feature Validation: Results**

An independent consistency check was performed on 19,776 transactions from 10 randomly sampled cards. The stored 24-hour transaction counts, sums, and means were compared against values reconstructed using only transactions occurring strictly before each transaction within the preceding 24 hours.

No discrepancies were identified across the three features. These findings support the correctness of the 24-hour rolling-window calculations and their alignment with individual transactions in the sampled data.

Although this check does not eliminate all possible sources of information leakage, it provides evidence that the exceptionally high Random Forest validation performance is not attributable to incorrect construction of these particular historical features.

### 3.3 CatBoost

CatBoost is a gradient-boosted decision tree algorithm that sequentially constructs trees to correct errors made by preceding iterations. Unlike Random Forest, which aggregates independently constructed trees, CatBoost uses boosting to progressively improve predictive performance.

CatBoost supports categorical predictors directly, eliminating the need for one-hot encoding, and handles missing numerical values natively. Consequently, numerical features are retained without scaling or imputation, while missing categorical values are replaced with an explicit category.

For the M0 baseline, CatBoost is configured with 300 boosting iterations, a learning rate of 0.05, and a maximum tree depth of 6. Class weighting is disabled to maintain comparability with the unweighted Logistic Regression and Random Forest baselines.

The model is trained on the same chronological training partition and evaluated using Average Precision (AP) and ROC-AUC on the validation partition.

In [26]:
from catboost import CatBoostClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
import time

# Prepare CatBoost-specific inputs
# Avoid copying the full numerical dataset unnecessarily
X_train_cb = X_train.copy()
X_val_cb = X_val.copy()

# CatBoost requires categorical columns to contain strings
for col in categorical_cols:
    X_train_cb[col] = X_train_cb[col].fillna("Missing").astype(str)
    X_val_cb[col] = X_val_cb[col].fillna("Missing").astype(str)

# Define unweighted CatBoost baseline
cb_model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    cat_features=categorical_cols,
    auto_class_weights=None,
    thread_count=2,
    random_seed=42,
    verbose=False,
    allow_writing_files=False
)

# Train model
start_time = time.time()

cb_model.fit(X_train_cb, y_train)

cb_training_time = time.time() - start_time

# Validation predictions
cb_val_probs = cb_model.predict_proba(X_val_cb)[:, 1]

cb_pr_auc = average_precision_score(y_val, cb_val_probs)
cb_roc_auc = roc_auc_score(y_val, cb_val_probs)

print(f"Training time: {cb_training_time:.2f} seconds")
print(f"Validation PR-AUC: {cb_pr_auc:.4f}")
print(f"Validation ROC-AUC: {cb_roc_auc:.4f}")

Training time: 307.41 seconds
Validation PR-AUC: 0.9791
Validation ROC-AUC: 0.9993


**CatBoost: Validation Results**

The unweighted CatBoost baseline achieved a validation Average Precision (AP) of **0.9791** and ROC-AUC of **0.9993**, outperforming both Logistic Regression and Random Forest. Its training time of **283.82 seconds** was lower than Random Forest's 435.28 seconds, although substantially higher than Logistic Regression's 26.33 seconds.

These findings suggest that gradient-boosted decision trees are effective at capturing complex relationships among transaction characteristics and historical spending behaviour. Nevertheless, the exceptionally high validation performance warrants continued attention to temporal validity and potential information leakage before drawing conclusions about deployment performance.

### 3.5 LightGBM

Light Gradient Boosting Machine (LightGBM) is a gradient-boosted decision tree algorithm that employs histogram-based learning and leaf-wise tree growth to improve computational efficiency while capturing nonlinear relationships and feature interactions.

LightGBM is included as an additional M0 baseline to compare its predictive performance and computational efficiency against CatBoost and Random Forest. This comparison is particularly relevant for subsequent temporal adaptation experiments, where repeated model retraining may introduce substantial computational overhead.

Numerical missing values are handled natively, while categorical variables are encoded using categories learned from the training partition. No class weighting or resampling is applied, preserving the unadjusted M0 setting.

The model is evaluated on the same chronological validation partition using Average Precision (AP), ROC-AUC and training time. The selected hyperparameters represent an initial baseline configuration rather than an optimised model.

In [27]:
import time
import lightgbm as lgb
from sklearn.metrics import average_precision_score, roc_auc_score

# Prepare categorical features using training-defined categories
X_train_lgb = X_train.copy()
X_val_lgb = X_val.copy()

for col in categorical_cols:
    X_train_lgb[col] = X_train_lgb[col].astype("category")

    X_val_lgb[col] = pd.Categorical(
        X_val_lgb[col],
        categories=X_train_lgb[col].cat.categories
    )

# Define unweighted LightGBM baseline
lgb_model = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    objective="binary",
    class_weight=None,
    is_unbalance=False,
    random_state=42,
    n_jobs=2,
    verbosity=-1
)

# Train
start_time = time.time()

lgb_model.fit(
    X_train_lgb,
    y_train,
    categorical_feature=categorical_cols
)

lgb_training_time = time.time() - start_time

# Predict and evaluate
lgb_val_probs = lgb_model.predict_proba(X_val_lgb)[:, 1]

lgb_pr_auc = average_precision_score(y_val, lgb_val_probs)
lgb_roc_auc = roc_auc_score(y_val, lgb_val_probs)

print(f"Training time: {lgb_training_time:.2f} seconds")
print(f"Validation PR-AUC: {lgb_pr_auc:.4f}")
print(f"Validation ROC-AUC: {lgb_roc_auc:.4f}")

Training time: 58.74 seconds
Validation PR-AUC: 0.9359
Validation ROC-AUC: 0.9990


### 3.5 Baseline Model Comparison

In [28]:
m0_results = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "PR-AUC (AP)": lr_pr_auc,
        "ROC-AUC": lr_roc_auc,
        "Training Time (s)": training_time
    },
    {
        "Model": "Random Forest",
        "PR-AUC (AP)": rf_pr_auc,
        "ROC-AUC": rf_roc_auc,
        "Training Time (s)": rf_training_time
    },
    {
        "Model": "CatBoost",
        "PR-AUC (AP)": cb_pr_auc,
        "ROC-AUC": cb_roc_auc,
        "Training Time (s)": cb_training_time
    },
    {
        "Model": "LightGBM",
        "PR-AUC (AP)": lgb_pr_auc,
        "ROC-AUC": lgb_roc_auc,
        "Training Time (s)": lgb_training_time
    }
])

display(
    m0_results
    .sort_values("PR-AUC (AP)", ascending=False)
    .round(4)
)

,Model,PR-AUC (AP),ROC-AUC,Training Time (s)
2,CatBoost,0.9791,0.9993,307.4056
1,Random Forest,0.9589,0.9981,404.3296
3,LightGBM,0.9359,0.9990,58.7439
0,Logistic Regression,0.6802,0.9664,21.4506



### Discussion

CatBoost achieved the strongest baseline performance, with the highest PR-AUC (0.9791) and ROC-AUC (0.9993), followed by Random Forest (PR-AUC = 0.9589) and LightGBM (PR-AUC = 0.9359). Logistic Regression performed substantially worse (PR-AUC = 0.6802), suggesting that nonlinear models are better suited to capturing the complex relationships and interactions associated with fraudulent transactions.

Although LightGBM achieved a slightly lower PR-AUC than CatBoost and Random Forest, it demonstrated substantially greater computational efficiency, requiring only 50.33 seconds to train compared with 293.34 seconds for CatBoost and 312.43 seconds for Random Forest. Its ROC-AUC of 0.9990 was also comparable to CatBoost, although its lower PR-AUC indicates weaker precision–recall performance.

Overall, CatBoost provides the strongest predictive baseline, while LightGBM offers a favourable trade-off between predictive performance and computational efficiency. These results establish the unweighted benchmarks against which M1 cost-sensitive learning will be evaluated. In M1, class weights will be introduced to address class imbalance while retaining the same chronological data partition and model configurations, allowing the effect of weighting to be assessed independently.



## 4. M1: Cost-Sensitive Learning


### 4.1 Methodology

M1 addresses severe class imbalance through inverse-frequency class weighting, assigning higher weights to minority-class fraudulent transactions to increase their contribution to the model's training objective.

Class weights are computed automatically from the training data using the models' built-in balanced weighting mechanisms. This increases the contribution of minority-class observations to the training objective without requiring resampling.

To isolate the effect of class weighting, M1 retains the same features, chronological training and validation partitions, preprocessing procedures, and model hyperparameters as M0. Performance is evaluated using PR-AUC, ROC-AUC, precision, recall, and F1-score.




### 4.2 Cost-Sensitive Logistic Regression

The M0 Logistic Regression baseline is extended by introducing inverse-frequency class weighting. Fraudulent transactions receive approximately 172.82 times the weight of legitimate transactions, increasing their contribution to the optimisation objective.

All preprocessing steps, model hyperparameters, and chronological data partitions are retained from M0 to isolate the effect of class weighting.


In [29]:
from sklearn.base import clone

# M1: Cost-sensitive Logistic Regression
lr_m1 = clone(lr_model)

lr_m1.set_params(
    classifier__class_weight="balanced"
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3I

In [30]:
import time
from sklearn.metrics import average_precision_score, roc_auc_score

start_time = time.time()

lr_m1.fit(X_train, y_train)

lr_m1_training_time = time.time() - start_time

# Predict fraud probabilities
lr_m1_probs = lr_m1.predict_proba(X_val)[:, 1]

# Evaluate
lr_m1_pr_auc = average_precision_score(y_val, lr_m1_probs)
lr_m1_roc_auc = roc_auc_score(y_val, lr_m1_probs)

print(f"Training time: {lr_m1_training_time:.2f} seconds")
print(f"PR-AUC (AP): {lr_m1_pr_auc:.4f}")
print(f"ROC-AUC: {lr_m1_roc_auc:.4f}")

Training time: 26.21 seconds
PR-AUC (AP): 0.4382
ROC-AUC: 0.9904



### 4.3 Cost-Sensitive Random Forest

The M0 Random Forest baseline is extended using `class_weight="balanced_subsample"`, which calculates inverse-frequency class weights separately for each bootstrap sample during tree construction. This increases the influence of minority-class fraudulent transactions while retaining the original model configuration, features, and chronological data partition.


In [31]:
from sklearn.base import clone
import time
from sklearn.metrics import average_precision_score, roc_auc_score

# M1: Cost-sensitive Random Forest
rf_m1 = clone(rf_model)

rf_m1.set_params(
    classifier__class_weight="balanced_subsample"
)

# Train
start_time = time.time()

rf_m1.fit(X_train, y_train)

rf_m1_training_time = time.time() - start_time

# Predict fraud probabilities
rf_m1_probs = rf_m1.predict_proba(X_val)[:, 1]

# Evaluate
rf_m1_pr_auc = average_precision_score(y_val, rf_m1_probs)
rf_m1_roc_auc = roc_auc_score(y_val, rf_m1_probs)

print(f"Training time: {rf_m1_training_time:.2f} seconds")
print(f"PR-AUC (AP): {rf_m1_pr_auc:.4f}")
print(f"ROC-AUC: {rf_m1_roc_auc:.4f}")

Training time: 323.11 seconds
PR-AUC (AP): 0.9562
ROC-AUC: 0.9981



### 4.4 Cost-Sensitive CatBoost

The M0 CatBoost baseline is extended using `auto_class_weights="Balanced"`, which automatically assigns inverse-frequency class weights to address the underrepresentation of fraudulent transactions.

The same features, chronological training and validation partitions, and model hyperparameters are retained from M0. This allows the effect of class weighting on predictive performance to be evaluated independently.


In [32]:
from catboost import CatBoostClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
import time

# M1: Cost-sensitive CatBoost
# Same M0 hyperparameters; only class weighting changes
cb_m1 = CatBoostClassifier(
    iterations=300,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    cat_features=categorical_cols,
    auto_class_weights="Balanced",
    thread_count=2,
    random_seed=42,
    verbose=False,
    allow_writing_files=False
)

# Train using the same CatBoost-specific inputs as M0
start_time = time.time()

cb_m1.fit(X_train_cb, y_train)

cb_m1_training_time = time.time() - start_time

# Validation predictions
cb_m1_probs = cb_m1.predict_proba(X_val_cb)[:, 1]

cb_m1_pr_auc = average_precision_score(y_val, cb_m1_probs)
cb_m1_roc_auc = roc_auc_score(y_val, cb_m1_probs)

print(f"Training time: {cb_m1_training_time:.2f} seconds")
print(f"PR-AUC (AP): {cb_m1_pr_auc:.4f}")
print(f"ROC-AUC: {cb_m1_roc_auc:.4f}")

Training time: 258.07 seconds
PR-AUC (AP): 0.9805
ROC-AUC: 0.9997



### 4.5 Cost-Sensitive LightGBM

The M0 LightGBM baseline is extended using `class_weight="balanced"`, which automatically assigns inverse-frequency weights to fraudulent and legitimate transactions.

The same features, chronological data partitions, and model hyperparameters are retained to isolate the effect of class weighting on predictive performance.


In [33]:
from sklearn.base import clone
from sklearn.metrics import average_precision_score, roc_auc_score
import time

# M1: Cost-sensitive LightGBM
lgb_m1 = clone(lgb_model)

lgb_m1.set_params(
    class_weight="balanced"
)

start_time = time.time()

lgb_m1.fit(X_train_lgb, y_train)

lgb_m1_training_time = time.time() - start_time

lgb_m1_probs = lgb_m1.predict_proba(X_val_lgb)[:, 1]

lgb_m1_pr_auc = average_precision_score(y_val, lgb_m1_probs)
lgb_m1_roc_auc = roc_auc_score(y_val, lgb_m1_probs)

print(f"Training time: {lgb_m1_training_time:.2f} seconds")
print(f"PR-AUC (AP): {lgb_m1_pr_auc:.4f}")
print(f"ROC-AUC: {lgb_m1_roc_auc:.4f}")

Training time: 35.48 seconds
PR-AUC (AP): 0.9860
ROC-AUC: 0.9996



### 4.6 M0–M1 Performance Comparison

To evaluate the effectiveness of cost-sensitive learning, the four M1 models are compared against their corresponding unweighted M0 baselines using the same chronological validation set.

Performance is assessed using PR-AUC (Average Precision) as the primary metric, alongside ROC-AUC, precision, recall, and F1-score. Precision, recall, and F1-score are calculated using a fixed classification threshold of 0.5 to examine how class weighting affects the trade-off between fraud detection and false positives.

By keeping the feature sets, data partitions, and model hyperparameters unchanged, the comparison isolates the effect of class weighting on predictive performance across different algorithms.


In [34]:
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

# Model, version, trained estimator, validation features
models = [
    ("Logistic Regression", "M0", lr_model, X_val),
    ("Logistic Regression", "M1", lr_m1, X_val),

    ("Random Forest", "M0", rf_model, X_val),
    ("Random Forest", "M1", rf_m1, X_val),

    ("CatBoost", "M0", cb_model, X_val_cb),
    ("CatBoost", "M1", cb_m1, X_val_cb),

    ("LightGBM", "M0", lgb_model, X_val_lgb),
    ("LightGBM", "M1", lgb_m1, X_val_lgb)
]

rows = []

for name, version, model, X in models:

    probs = model.predict_proba(X)[:, 1]
    predictions = (probs >= 0.5).astype(int)

    rows.append({
        "Model": name,
        "Version": version,
        "PR-AUC": average_precision_score(y_val, probs),
        "ROC-AUC": roc_auc_score(y_val, probs),
        "Precision": precision_score(y_val, predictions, zero_division=0),
        "Recall": recall_score(y_val, predictions, zero_division=0),
        "F1": f1_score(y_val, predictions, zero_division=0)
    })

comparison_df = pd.DataFrame(rows)

display(comparison_df.round(4))

,Model,Version,PR-AUC,ROC-AUC,Precision,Recall,F1
0,Logistic Regression,M0,0.6802,0.9664,0.8820,0.5007,0.6387
1,Logistic Regression,M1,0.4382,0.9904,0.1734,0.9486,0.2933
2,Random Forest,M0,0.9589,0.9981,0.9835,0.8160,0.8920
3,Random Forest,M1,0.9562,0.9981,0.7904,0.9389,0.8582
4,CatBoost,M0,0.9791,0.9993,0.9827,0.9239,0.9524
5,CatBoost,M1,0.9805,0.9997,0.6059,0.9857,0.7505
6,LightGBM,M0,0.9359,0.9990,0.9540,0.9174,0.9354
7,LightGBM,M1,0.9860,0.9996,0.8671,0.9753,0.9180



#### 4.6.1 Results and Discussion

Cost-sensitive learning increased fraud recall across all four models, although its effects on precision, F1-score, and PR-AUC varied considerably.

**Logistic Regression** exhibited the largest increase in recall, from 0.5007 to 0.9486. However, precision declined sharply from 0.8820 to 0.1734, while PR-AUC decreased from 0.6802 to 0.4382. This suggests that class weighting substantially increased the model's sensitivity to fraud but reduced its ability to distinguish fraudulent transactions without generating false positives at the fixed classification threshold.

**Random Forest** achieved higher recall (0.8160 to 0.9389), accompanied by lower precision (0.9835 to 0.7904). Its PR-AUC remained relatively stable, decreasing marginally from 0.9589 to 0.9562, indicating limited improvement in overall ranking performance.

**CatBoost** achieved a marginal improvement in PR-AUC from 0.9791 to 0.9805, alongside an increase in recall from 0.9239 to 0.9857. However, precision decreased substantially from 0.9827 to 0.6059, reducing its F1-score from 0.9524 to 0.7505.

**LightGBM** demonstrated the largest improvement in PR-AUC, increasing from 0.9359 to 0.9860, and achieved the highest PR-AUC among all evaluated models. Its recall increased from 0.9174 to 0.9753, while precision declined moderately from 0.9540 to 0.8671.

Overall, the results demonstrate that inverse-frequency class weighting consistently improves fraud recall but does not necessarily improve precision or F1-score at a fixed threshold of 0.5. Although weighted LightGBM achieved the highest PR-AUC, unweighted CatBoost retained the highest F1-score. These findings highlight the importance of evaluating both threshold-independent ranking performance and threshold-dependent classification outcomes when assessing fraud detection models.


## 5. M2: Temporal Adaptation Through Sliding-Window Retraining

### 5.1 Methodology and Experimental Design

M2 evaluates whether periodic retraining enables fraud detection models to adapt to evolving transaction patterns. Unlike static M0/M1 models, which are trained once, M2 periodically updates model parameters using newly verified transaction labels.

The experiment incorporates a seven-day label verification delay. At each retraining cutoff, transactions from the preceding seven days cannot contribute their labels to supervised training. However, their observable transaction attributes may still contribute to historical features.

Four model families are evaluated: Logistic Regression, Random Forest, CatBoost and LightGBM. Each retains its better-performing M0 or M1 configuration.

**Development-stage configuration selection**

- Training windows: 28-day rolling, 56-day rolling and expanding history.
- Retraining frequencies: Every 7 or 14 days.
- Label verification delay: Seven days.
- Evaluation: Chronological development blocks within the original training partition.
- Primary metric: Pooled PR-AUC (Average Precision), supplemented by ROC-AUC.

**Validation-stage evaluation**

For each model family, the selected adaptive configuration is compared against a static model on the same chronological validation transactions. Both use identical initial training windows and label availability constraints; only the adaptive model periodically retrains.

A separate zero-day versus seven-day label-delay experiment measures the performance cost of delayed verification. Weekly PR-AUC is also examined to assess whether adaptive retraining maintains performance over time.

The final test partition remains untouched until model selection is complete. Improvements from M2 are interpreted as evidence of the usefulness of adaptation; concept drift is investigated separately through temporal feature-distribution and performance analysis.

### 5.2 Chronological Data Preparation and Leakage Controls

The processed Sparkov dataset is reloaded because Section 2 released the full DataFrame after constructing the original training and validation subsets.

Transactions are ordered chronologically, and only the original training and validation partitions are retained for M2 development. The held-out test partition is excluded.

Historical frequency features are reconstructed using strictly earlier transactions, ensuring that each transaction's frequency counts reflect information observable at its timestamp. Transactions sharing the same timestamp do not contribute to one another's history.

This preparation is shared across all four model families and all M2 experiments. The remaining historical features are inherited from the existing feature-engineering pipeline and require separate leakage verification.

In [49]:

import numpy as np
import pandas as pd
from pathlib import Path

# Reload processed data for chronological M2 experiments
m2_df = pd.read_pickle(
    Path("../data/processed/sparkov_features.pkl")
)

m2_df["trans_date_trans_time"] = pd.to_datetime(
    m2_df["trans_date_trans_time"]
)

# Exclude the held-out test partition
m2_df = m2_df.loc[
    m2_df["sparkov_split"].isin(["train", "validation"])
].copy()

# Sort chronologically, preserving deterministic row ordering
m2_df = m2_df.sort_values(
    "trans_date_trans_time", kind="stable"
).reset_index(drop=True)

# Recalculate frequency features from strictly earlier timestamps
frequency_sources = {
    "merchant_freq": "merchant",
    "category_freq": "category",
    "state_freq": "state"
}

for feature, source in frequency_sources.items():
    if feature not in feature_cols:
        continue

    previous_rows = m2_df.groupby(
        source, dropna=False
    ).cumcount()

    same_timestamp_previous = m2_df.groupby(
        [source, "trans_date_trans_time"],
        dropna=False
    ).cumcount()

    m2_df[feature] = (
        previous_rows - same_timestamp_previous
    ).astype("int64")

# Basic integrity checks
assert m2_df["trans_date_trans_time"].is_monotonic_increasing
assert m2_df[feature_cols].shape[1] == len(feature_cols)
assert set(m2_df["sparkov_split"].unique()) == {
    "train", "validation"
}

for feature in frequency_sources:
    if feature in feature_cols:
        assert (m2_df[feature] >= 0).all()

print("M2 data prepared successfully.")
print("Shape:", m2_df.shape)
print("\nChronological partitions:")
print(
    m2_df.groupby("sparkov_split").agg(
        rows=("is_fraud", "size"),
        fraud_rate=("is_fraud", "mean"),
        first_transaction=("trans_date_trans_time", "min"),
        last_transaction=("trans_date_trans_time", "max")
    )
)


M2 data prepared successfully.
Shape: (1296675, 87)

Chronological partitions:
                  rows  fraud_rate   first_transaction    last_transaction
sparkov_split                                                             
train          1037340    0.005753 2019-01-01 00:00:18 2020-03-06 07:15:17
validation      259335    0.005931 2020-03-06 07:16:43 2020-06-21 12:13:37


### 5.3 Training Windows and Label Availability

M2 simulates periodic retraining under delayed fraud-label verification. At each retraining cutoff, only transactions whose labels have become available can be used for supervised training.

Three training-history strategies are supported:

- **28-day rolling:** Uses the most recent 28 days of verified transactions.
- **56-day rolling:** Uses the most recent 56 days of verified transactions.
- **Expanding:** Uses all eligible historical transactions.

The label verification delay is configurable, with seven days used in the main experiment and zero days used for sensitivity analysis.

During development, training is restricted to the original training partition. During validation, previously verified validation transactions may also be incorporated into subsequent retraining, simulating a continuously operating fraud detection system.

The final test partition is excluded throughout M2 development and validation.

In [50]:

# M2 experiment settings
M2_WINDOWS = [28, 56, None]  # None = expanding
M2_RETRAIN_FREQUENCIES = [7, 14]
M2_LABEL_DELAY_DAYS = 7

M2_TIME_COL = "trans_date_trans_time"
M2_TARGET_COL = "is_fraud"


def get_m2_training_data(
    df,
    cutoff,
    window_days=None,
    label_delay_days=7,
    allowed_splits=("train",)
):
    """
    Select labelled transactions available at a retraining cutoff.

    window_days:
        28 or 56 = rolling training window
        None = expanding history

    label_delay_days:
        Number of days before a transaction label becomes available.

    allowed_splits:
        Development: ("train",)
        Validation: ("train", "validation")
    """
    cutoff = pd.Timestamp(cutoff)

    label_cutoff = cutoff - pd.Timedelta(
        days=label_delay_days
    )

    mask = (
        df["sparkov_split"].isin(allowed_splits)
        & (df[M2_TIME_COL] < label_cutoff)
    )

    if window_days is not None:
        window_start = label_cutoff - pd.Timedelta(
            days=window_days
        )
        mask &= df[M2_TIME_COL] >= window_start

    return df.loc[mask].copy()


# Sanity check at a cutoff inside the training partition
m2_test_cutoff = pd.Timestamp("2019-08-01")

for window in M2_WINDOWS:
    sample = get_m2_training_data(
        m2_df,
        cutoff=m2_test_cutoff,
        window_days=window,
        label_delay_days=7,
        allowed_splits=("train",)
    )

    print(
        f"Window: {str(window):>4} | "
        f"Rows: {len(sample):>8,} | "
        f"Fraud: {sample[M2_TARGET_COL].sum():>5,} | "
        f"Latest eligible: {sample[M2_TIME_COL].max()}"
    )

    assert (
        sample[M2_TIME_COL]
        < m2_test_cutoff - pd.Timedelta(days=7)
    ).all()

print("\nTraining-window checks passed.")


Window:   28 | Rows:   78,657 | Fraud:   313 | Latest eligible: 2019-07-24 23:59:12
Window:   56 | Rows:  157,521 | Fraud:   588 | Latest eligible: 2019-07-24 23:59:12
Window: None | Rows:  466,773 | Fraud: 2,881 | Latest eligible: 2019-07-24 23:59:12

Training-window checks passed.


### 5.4 Reusable Model Configurations

Four classifier families are evaluated under the same M2 temporal adaptation framework. Each retains its better-performing configuration from the M0/M1 experiments, without introducing additional hyperparameter tuning.

Logistic Regression and Random Forest use their unweighted M0 configurations. CatBoost and LightGBM use their cost-sensitive M1 configurations.

A shared model factory constructs a fresh classifier for every retraining cycle. Model-specific preprocessing is handled consistently: Logistic Regression and Random Forest use their existing scikit-learn preprocessing pipelines, CatBoost uses string-valued categorical features, and LightGBM uses training-defined categorical levels.

This ensures that the subsequent M2 experiments vary the temporal training strategy rather than the underlying model configuration.

In [51]:

from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from catboost import CatBoostClassifier
import lightgbm as lgb


M2_MODEL_NAMES = [
    "Logistic Regression",
    "Random Forest",
    "CatBoost",
    "LightGBM"
]


def make_m2_model(model_name):
    """Create a fresh model using the selected M0/M1 configuration."""

    if model_name == "Logistic Regression":
        return Pipeline([
            ("preprocessor", clone(lr_preprocessor)),
            ("classifier", LogisticRegression(
                max_iter=1000,
                class_weight=None,
                random_state=42
            ))
        ])

    if model_name == "Random Forest":
        return Pipeline([
            ("preprocessor", clone(rf_preprocessor)),
            ("classifier", RandomForestClassifier(
                n_estimators=100,
                max_depth=15,
                min_samples_leaf=10,
                max_features="sqrt",
                class_weight=None,
                n_jobs=2,
                random_state=42
            ))
        ])

    if model_name == "CatBoost":
        return CatBoostClassifier(
            iterations=300,
            learning_rate=0.05,
            depth=6,
            loss_function="Logloss",
            cat_features=categorical_cols,
            auto_class_weights="Balanced",
            thread_count=2,
            random_seed=42,
            verbose=False,
            allow_writing_files=False
        )

    if model_name == "LightGBM":
        return lgb.LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            max_depth=-1,
            objective="binary",
            class_weight="balanced",
            is_unbalance=False,
            random_state=42,
            n_jobs=2,
            verbosity=-1
        )

    raise ValueError(f"Unknown model: {model_name}")


def prepare_m2_features(frame, model_name, categories=None):
    """
    Prepare features for the requested model.

    For LightGBM, pass training categories to transform scoring data
    using the same category definitions.
    """
    X = frame[feature_cols].copy()

    if model_name == "CatBoost":
        for col in categorical_cols:
            X[col] = X[col].fillna("Missing").astype(str)

    elif model_name == "LightGBM":
        for col in categorical_cols:
            if categories is None:
                X[col] = X[col].astype("category")
            else:
                X[col] = pd.Categorical(
                    X[col],
                    categories=categories[col]
                )

    return X


# Check that all four models can be constructed
for name in M2_MODEL_NAMES:
    model = make_m2_model(name)
    print(f"{name}: {type(model).__name__}")

print("\nAll M2 model configurations created successfully.")


Logistic Regression: Pipeline
Random Forest: Pipeline
CatBoost: CatBoostClassifier
LightGBM: LGBMClassifier

All M2 model configurations created successfully.


### 5.5 Unified Adaptive Retraining and Evaluation

A reusable evaluation engine simulates fraud detection over chronological scoring blocks.

At each retraining cutoff, the model is fitted using only labels available after the specified verification delay. The fitted model then scores subsequent transactions until the next scheduled retraining cutoff.

The engine supports different model families, rolling or expanding training histories, retraining frequencies, and label delays. Predictions are retained at transaction level to support pooled PR-AUC, ROC-AUC, and weekly performance comparisons.

For LightGBM, categorical levels are established using the current training window and reused when transforming the scoring block. All model families use their predefined M0/M1 configurations.

The same engine will subsequently be used for development-stage configuration selection and validation-stage adaptive evaluation.

In [52]:

from sklearn.metrics import average_precision_score, roc_auc_score


def run_m2_adaptive(
    df,
    model_name,
    start,
    end,
    window_days=None,
    retrain_every_days=7,
    label_delay_days=7,
    allowed_splits=("train",),
    scoring_split="train"
):
    """
    Chronological adaptive retraining with transaction-level predictions.

    Only scoring rows in [start, end) are evaluated.
    Retraining uses labels available at each cutoff.
    """
    start = pd.Timestamp(start)
    end = pd.Timestamp(end)

    if start >= end:
        raise ValueError("start must be earlier than end")

    predictions = []
    cutoffs = pd.date_range(
        start=start,
        end=end,
        freq=f"{retrain_every_days}D",
        inclusive="left"
    )

    for i, cutoff in enumerate(cutoffs):
        next_cutoff = min(
            cutoff + pd.Timedelta(days=retrain_every_days),
            end
        )

        train_block = get_m2_training_data(
            df,
            cutoff=cutoff,
            window_days=window_days,
            label_delay_days=label_delay_days,
            allowed_splits=allowed_splits
        )

        score_block = df.loc[
            df["sparkov_split"].eq(scoring_split)
            & df[M2_TIME_COL].ge(cutoff)
            & df[M2_TIME_COL].lt(next_cutoff)
        ].copy()

        if score_block.empty:
            continue

        if train_block.empty or train_block[M2_TARGET_COL].nunique() < 2:
            raise ValueError(
                f"Insufficient training classes at cutoff {cutoff}"
            )

        model = make_m2_model(model_name)

        X_train = prepare_m2_features(train_block, model_name)

        if model_name == "LightGBM":
            categories = {
                col: X_train[col].cat.categories
                for col in categorical_cols
            }
            X_score = prepare_m2_features(
                score_block, model_name, categories=categories
            )
        else:
            X_score = prepare_m2_features(score_block, model_name)

        model.fit(X_train, train_block[M2_TARGET_COL])

        probabilities = model.predict_proba(X_score)[:, 1]

        block_predictions = pd.DataFrame({
            "transaction_index": score_block.index,
            "timestamp": score_block[M2_TIME_COL].to_numpy(),
            "y_true": score_block[M2_TARGET_COL].to_numpy(),
            "y_score": probabilities,
            "cutoff": cutoff,
            "model": model_name,
            "window_days": (
                "expanding" if window_days is None else str(window_days)
            ),
            "retrain_every_days": retrain_every_days,
            "label_delay_days": label_delay_days
        })

        predictions.append(block_predictions)

    if not predictions:
        raise ValueError("No scoring transactions found")

    result = pd.concat(predictions, ignore_index=True)

    if result["transaction_index"].duplicated().any():
        raise AssertionError("Duplicate scored transactions")

    return result


def summarize_m2_predictions(predictions):
    """Calculate pooled metrics across all scored transactions."""
    y_true = predictions["y_true"]
    y_score = predictions["y_score"]

    return {
        "transactions": len(predictions),
        "fraud_cases": int(y_true.sum()),
        "pr_auc": average_precision_score(y_true, y_score),
        "roc_auc": (
            roc_auc_score(y_true, y_score)
            if y_true.nunique() == 2 else np.nan
        )
    }
